# Task 1 — Hindi Punctuation Restoration (Kaggle)

**Model:** `distilbert-base-multilingual-cased` (~135M params) fine-tuned for token classification
**Data:** AI4Bharat Sangraha (`verified/hin`), streamed from Hugging Face
**Labels:** `O` (none) · `END` (।) · `COMMA` (,) · `QUESTION` (?)

### Before you run (Notebook settings, right panel)
1. **Accelerator → GPU T4 x2** (don't use P100; newer PyTorch builds don't support it). This notebook uses only 1 GPU so the memory numbers stay honest.
2. **Internet → On** (needs phone verification on your Kaggle account).
3. **Add your sample files** → *Add Input → Upload → New Dataset*, upload `Sample-Raw-No-Punctuations.zip` (Kaggle unzips it automatically).
   *Or* skip this and use the Google Drive option in Cell 2.

Then: **Run All**. Everything is saved in `/kaggle/working`. When it finishes, click **Save Version → Save & Run All** so the outputs stay saved permanently.

In [ ]:
# ===== CELL 1: Config + environment check =====
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"      # use ONE GPU only (clean memory measurement)
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import random, re, json, glob, shutil, itertools, collections
import numpy as np
import torch

SEED        = 42
MODEL_NAME  = "distilbert-base-multilingual-cased"
NUM_DOCS    = 8000      # Sangraha documents to use (increase to 5000 if you have time)
CHUNK_WORDS = 64        # words per training example
MAX_LEN     = 192       # max subword tokens per example
EPOCHS      = 3
WORK        = "/kaggle/working"
DATA_DIR    = f"{WORK}/data"
MODEL_DIR   = f"{WORK}/punct_model"
OUT_DIR     = f"{WORK}/outputs"
for d in [DATA_DIR, MODEL_DIR, OUT_DIR, f"{OUT_DIR}/punctuated"]:
    os.makedirs(d, exist_ok=True)

LABELS   = ["O", "END", "COMMA", "QUESTION"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
PUNCT_OUT = {"O": "", "END": "।", "COMMA": ",", "QUESTION": "?"}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Turn on GPU T4 x2 in Notebook settings before training.")

In [ ]:
# ===== CELL 2: Get the sample files (Kaggle dataset OR Google Drive) =====
SAMPLE_DIR = f"{WORK}/samples"
os.makedirs(SAMPLE_DIR, exist_ok=True)

# --- Option A: you added the ZIP as a Kaggle dataset (recommended) ---
found = sorted(glob.glob("/kaggle/input/**/sample-*.txt", recursive=True))

# --- Option B: download from Google Drive ---
# In Drive: right-click the ZIP -> Share -> "Anyone with the link" -> Copy link, paste below.
GDRIVE_ZIP_LINK = ""   # e.g. "https://drive.google.com/file/d/XXXXXXXX/view?usp=sharing"

if not found and GDRIVE_ZIP_LINK:
    !pip -q install gdown
    import gdown, zipfile
    zip_path = f"{WORK}/samples.zip"
    gdown.download(GDRIVE_ZIP_LINK, zip_path, fuzzy=True, quiet=False)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(SAMPLE_DIR)
    found = sorted(glob.glob(f"{SAMPLE_DIR}/**/sample-*.txt", recursive=True))

assert found, "No sample-*.txt files found. Add the ZIP as a Kaggle dataset or set GDRIVE_ZIP_LINK."

def sample_key(p):
    m = re.search(r"sample-(\d+)", os.path.basename(p))
    return int(m.group(1)) if m else 999

SAMPLE_FILES = sorted(set(found), key=sample_key)
# keep one file per name (in case the zip had duplicates)
_seen = {}
for p in SAMPLE_FILES:
    _seen.setdefault(os.path.basename(p), p)
SAMPLE_FILES = sorted(_seen.values(), key=sample_key)

for p in SAMPLE_FILES:
    print(os.path.basename(p), "->", p)

In [ ]:
# ===== CELL 3: Look at the samples =====
for p in SAMPLE_FILES:
    txt = open(p, encoding="utf-8").read()
    print(f"{os.path.basename(p)}: {len(txt.split())} words, NULL appears {txt.count('NULL')} times")
print("\nPreview of first file:\n", open(SAMPLE_FILES[0], encoding="utf-8").read()[:400])

In [ ]:
# ===== CELL 4: Stream Sangraha Hindi (verified) from Hugging Face =====
from datasets import load_dataset

# Optional: if Hugging Face asks for login, add a Kaggle secret named HF_TOKEN (Add-ons -> Secrets)
try:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    login(UserSecretsClient().get_secret("HF_TOKEN"))
    print("Logged in to Hugging Face")
except Exception:
    print("No HF_TOKEN secret, continuing without login")

stream = load_dataset("ai4bharat/sangraha", data_dir="verified/hin", split="train", streaming=True)
stream = stream.shuffle(seed=SEED, buffer_size=10_000)

docs = []
for ex in stream:
    t = (ex.get("text") or "").strip()
    # keep real documents that actually contain Hindi punctuation
    if len(t.split()) >= 40 and ("।" in t or "?" in t):
        docs.append(t)
    if len(docs) >= NUM_DOCS:
        break

print("Documents collected:", len(docs))
print("\nExample:\n", docs[0][:500])

In [ ]:
# ===== CELL 5: Turn punctuated text into (words, labels) =====
# Every word gets a label = the punctuation that comes right after it.
PUNCT_TO_LABEL = {"।": "END", "॥": "END", ".": "END", "!": "END",
                  "?": "QUESTION", ",": "COMMA", ";": "COMMA"}
PRIORITY = {"O": 0, "COMMA": 1, "END": 2, "QUESTION": 3}

# numbers like 3.5 or 1,000 stay one word; then normal words; then punctuation marks
TOKEN_RE = re.compile(r"\d+(?:[.,]\d+)+|[^\s।॥.,?!;]+|[।॥.,?!;]")
STRIP_CHARS = "\"'“”‘’()[]{}<>-–—…:|*#/\\`~_"

def clean_word(w):
    return w.strip(STRIP_CHARS)

def text_to_words_labels(text):
    words, labels = [], []
    for tok in TOKEN_RE.findall(text):
        if tok in PUNCT_TO_LABEL:
            if words:
                new = PUNCT_TO_LABEL[tok]
                if PRIORITY[new] > PRIORITY[labels[-1]]:
                    labels[-1] = new
        else:
            w = clean_word(tok)
            if w:
                words.append(w); labels.append("O")
    return words, labels

def raw_to_words(text, drop_null=True):
    # for the unpunctuated samples
    if drop_null:
        text = re.sub(r"\bNULL\b", " ", text)
    return [w for w in (clean_word(t) for t in TOKEN_RE.findall(text) if t not in PUNCT_TO_LABEL) if w]

# quick test
demo = "राम घर गया। क्या तुम आओगे? हाँ, मैं आऊँगा।"
for w, l in zip(*text_to_words_labels(demo)):
    print(f"{w:10s} -> {l}")

In [ ]:
# ===== CELL 6: 80:20 document-level split + chunking + save =====
rng = random.Random(SEED)
idx = list(range(len(docs)))
rng.shuffle(idx)
cut = int(0.8 * len(idx))
train_docs = [docs[i] for i in idx[:cut]]
test_docs  = [docs[i] for i in idx[cut:]]

def make_chunks(doc_list):
    chunks = []
    for d in doc_list:
        w, l = text_to_words_labels(d)
        for s in range(0, len(w), CHUNK_WORDS):
            cw, cl = w[s:s+CHUNK_WORDS], l[s:s+CHUNK_WORDS]
            if len(cw) >= 8:
                chunks.append({"words": cw, "labels": cl})
    return chunks

train_chunks = make_chunks(train_docs)
test_chunks  = make_chunks(test_docs)

for name, ch in [("train", train_chunks), ("test", test_chunks)]:
    with open(f"{DATA_DIR}/{name}.jsonl", "w", encoding="utf-8") as f:
        for c in ch:
            f.write(json.dumps(c, ensure_ascii=False) + "\n")

cnt = collections.Counter(l for c in train_chunks for l in c["labels"])
print(f"Train docs: {len(train_docs)} -> {len(train_chunks)} chunks")
print(f"Test  docs: {len(test_docs)} -> {len(test_chunks)} chunks")
print("Train label counts:", dict(cnt))

In [ ]:
# ===== CELL 7: Tokenize + align labels to subwords =====
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align(batch):
    enc = tokenizer(batch["words"], is_split_into_words=True,
                    truncation=True, max_length=MAX_LEN)
    all_labels = []
    for i, labs in enumerate(batch["labels"]):
        word_ids = enc.word_ids(batch_index=i)
        prev, ids = None, []
        for wid in word_ids:
            if wid is None or wid == prev:
                ids.append(-100)            # special tokens / later subwords: ignored
            else:
                ids.append(label2id[labs[wid]])   # label only the FIRST subword of each word
            prev = wid
        all_labels.append(ids)
    enc["labels"] = all_labels
    return enc

train_ds = Dataset.from_list(train_chunks).map(tokenize_and_align, batched=True, remove_columns=["words", "labels"])
test_ds  = Dataset.from_list(test_chunks).map(tokenize_and_align, batched=True, remove_columns=["words", "labels"])
print(train_ds, test_ds)

In [ ]:
# ===== CELL 8: Load model + freeze embeddings (saves GPU memory) =====
from transformers import AutoModelForTokenClassification

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

# The embedding table is ~92M of the ~135M params. Freezing it means no gradients
# or Adam states for it -> training fits under 1.5 GB.
for p in model.distilbert.embeddings.parameters():
    p.requires_grad = False

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total/1e6:.1f}M | Trainable: {trainable/1e6:.1f}M")

In [ ]:
# ===== CELL 9: Train (weighted loss because most words have no punctuation) =====
from transformers import Trainer, TrainingArguments, DataCollatorForTokenClassification
from sklearn.metrics import f1_score

# class weights: rarer labels count more
counts = np.array([cnt.get(l, 1) for l in LABELS], dtype=np.float64)
weights = np.sqrt(counts.sum() / (len(LABELS) * counts))
weights = np.clip(weights / weights[0], 1.0, 5.0)
print("Class weights:", dict(zip(LABELS, weights.round(2))))
class_w = torch.tensor(weights, dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_w.to(logits.device), ignore_index=-100)
        loss = loss_fn(logits.view(-1, logits.size(-1)).float(), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(-1)
    mask = labels != -100
    y_true, y_pred = labels[mask], preds[mask]
    return {"f1_punct": f1_score(y_true, y_pred, labels=[1, 2, 3], average="macro", zero_division=0)}

args = TrainingArguments(
    output_dir=f"{WORK}/checkpoints",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,      # effective batch 16
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
        warmup_steps=int(0.1 * EPOCHS * (len(train_ds) // 16 + 1)),
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_punct",
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=test_ds,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

trainer.train()

train_peak_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
train_reserved_gb = torch.cuda.max_memory_reserved() / 1024**3 if torch.cuda.is_available() else 0.0
print(f"\nPeak GPU memory during TRAINING: {train_peak_gb:.3f} GB allocated ({train_reserved_gb:.3f} GB reserved)")
print("UNDER 1.5 GB" if train_peak_gb < 1.5 else "OVER 1.5 GB -> set per_device_train_batch_size=4 and rerun")

In [ ]:
# ===== CELL 10: Evaluate on the 20% test split =====
from sklearn.metrics import classification_report, confusion_matrix

pred_out = trainer.predict(test_ds)
preds = pred_out.predictions.argmax(-1)
labels = pred_out.label_ids
mask = labels != -100
y_true, y_pred = labels[mask], preds[mask]

report_txt = classification_report(y_true, y_pred, labels=[0, 1, 2, 3],
                                   target_names=LABELS, digits=4, zero_division=0)
print(report_txt)
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
print("Confusion matrix (rows = true, cols = predicted):", LABELS)
print(cm)

with open(f"{OUT_DIR}/eval_report.txt", "w", encoding="utf-8") as f:
    f.write(report_txt + "\n\nConfusion matrix (rows=true, cols=pred) " + str(LABELS) + "\n" + str(cm))

In [ ]:
# ===== CELL 11: Save the model (this is "Model 1" for Drive) =====
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
shutil.make_archive(f"{WORK}/punct_model", "zip", MODEL_DIR)
print("Model saved to", MODEL_DIR)
print("Zip:", f"{WORK}/punct_model.zip", f"({os.path.getsize(f'{WORK}/punct_model.zip')/1024**2:.0f} MB)")

In [ ]:
# ===== CELL 12: Punctuate the 7 samples =====
model = trainer.model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

WIN, STRIDE = CHUNK_WORDS, 48   # overlapping windows: each word gets right-side context

@torch.no_grad()
def predict_words(words):
    preds = ["O"] * len(words)
    start = 0
    while start < len(words):
        win = words[start:start+WIN]
        enc = tokenizer(win, is_split_into_words=True, truncation=True,
                        max_length=MAX_LEN, return_tensors="pt")
        word_ids = enc.word_ids(0)
        logits = model(**{k: v.to(device) for k, v in enc.items()}).logits[0]
        ids = logits.argmax(-1).tolist()
        last = start + WIN >= len(words)
        prev = None
        for pos, wid in enumerate(word_ids):
            if wid is not None and wid != prev:
                if last or wid < STRIDE:          # this window "owns" its first STRIDE words
                    preds[start + wid] = id2label[ids[pos]]
            prev = wid
        if last:
            break
        start += STRIDE
    return preds

def punctuate(words, preds):
    out = []
    for w, p in zip(words, preds):
        out.append(w + PUNCT_OUT[p])
    text = " ".join(out)
    if preds and preds[-1] == "O":
        text += "।"
    return text

if torch.cuda.is_available():
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

results = {}
for p in SAMPLE_FILES:
    name = os.path.basename(p)
    words = raw_to_words(open(p, encoding="utf-8").read(), drop_null=True)
    preds = predict_words(words)
    text = punctuate(words, preds)
    results[name] = text
    with open(f"{OUT_DIR}/punctuated/{name}", "w", encoding="utf-8") as f:
        f.write(text)
    c = collections.Counter(preds)
    print(f"{name}: {len(words)} words | END={c['END']} COMMA={c['COMMA']} QUESTION={c['QUESTION']}")

infer_peak_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else 0.0
print(f"\nPeak GPU memory during INFERENCE: {infer_peak_gb:.3f} GB")

In [ ]:
# ===== CELL 13: Preview + memory report + zip outputs =====
first = list(results)[0]
print(f"--- {first} (first 800 chars) ---\n{results[first][:800]}\n")

mem_report = {
    "model": MODEL_NAME,
    "total_params_M": round(total/1e6, 1),
    "trainable_params_M": round(trainable/1e6, 1),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "train_peak_gpu_gb": round(train_peak_gb, 3),
    "train_reserved_gpu_gb": round(train_reserved_gb, 3),
    "inference_peak_gpu_gb": round(infer_peak_gb, 3),
    "under_1_5_gb": bool(train_peak_gb < 1.5 and infer_peak_gb < 1.5),
    "num_docs": len(docs), "train_chunks": len(train_chunks), "test_chunks": len(test_chunks),
    "seed": SEED, "epochs": EPOCHS,
}
with open(f"{OUT_DIR}/memory_report.json", "w") as f:
    json.dump(mem_report, f, indent=2)
print(json.dumps(mem_report, indent=2))

shutil.make_archive(f"{WORK}/task1_outputs", "zip", OUT_DIR)
shutil.rmtree(f"{WORK}/checkpoints", ignore_errors=True)   # free disk; final model is in punct_model/
print("\nDone! Download from the Output panel:")
print(" - punct_model.zip     -> upload to Google Drive (Model 1)")
print(" - task1_outputs.zip   -> punctuated samples + eval_report.txt + memory_report.json")

In [ ]:
from IPython.display import FileLink
FileLink("punct_model.zip")